In [1]:
# === Cell 0: 環境準備 ===
import os, sys, shutil
REPO_DIR = '/content/Data_Mining'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/eric20041027/Data_Mining.git $REPO_DIR
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# 一次裝完今天會用到的所有套件（含 DeBERTa 需要的 sentencepiece）
!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "datasets>=2.20" "scikit-learn>=1.4" "sentencepiece"

Cloning into '/content/Data_Mining'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 97 (delta 46), reused 60 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 6.30 MiB | 19.16 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Mounted at /content/drive
CUDA: True
Device: NVIDIA A100-SXM4-40GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 160.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.6 MB/s eta 0:00:00


In [2]:
# === Cell 1: 從 Drive 還原 v14 backup ===
import tarfile, os

tar_path = '/content/drive/MyDrive/Kaggle_backup/predictions_v14.tar.gz'
assert os.path.exists(tar_path), f'❌ Backup 不存在！檢查 Drive 路徑'

print(f'Restoring from {tar_path}...')
with tarfile.open(tar_path) as tar:
    tar.extractall('/content/Data_Mining')

src = '/content/Data_Mining/outputs/bert_runs'
dirs = sorted(d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d)) and d != 'smoke_test')
print(f'\n還原後 {len(dirs)} 個 run 資料夾：')
for d in dirs:
    print(f'  {d}')

# 確認應該看到的：
# - 5 個 biobert_noweight_seed42_fold[0-4]
# - 5 個 pubmedbert_base_seed2024_fold[0-4]  (balanced 舊版)
# - 5 個 pubmedbert_base_seed42_fold[0-4]    (balanced 舊版)
# - 5 個 pubmedbert_noweight_seed42_fold[0-4]
# - 5 個 pubmedbert_noweight_seed2024_fold[0-4]
# - 5 個 pubmedbert_noweight_seed7_fold[0-4]
# - 5 個 pubmedbertlarge_noweight_seed42_fold[0-4]
# - 5 個 biobert_base_seed42_fold[0-4]       (balanced 舊版)
# 預期共 35～40 個（端看 v14 backup 包了多少）

Restoring from /content/drive/MyDrive/Kaggle_backup/predictions_v14.tar.gz...

還原後 40 個 run 資料夾：
  biobert_base_seed42_fold0
  biobert_base_seed42_fold1
  biobert_base_seed42_fold2
  biobert_base_seed42_fold3
  biobert_base_seed42_fold4
  biobert_noweight_seed42_fold0
  biobert_noweight_seed42_fold1
  biobert_noweight_seed42_fold2
  biobert_noweight_seed42_fold3
  biobert_noweight_seed42_fold4
  pubmedbert_base_seed2024_fold0
  pubmedbert_base_seed2024_fold1
  pubmedbert_base_seed2024_fold2
  pubmedbert_base_seed2024_fold3
  pubmedbert_base_seed2024_fold4
  pubmedbert_base_seed42_fold0
  pubmedbert_base_seed42_fold1
  pubmedbert_base_seed42_fold2
  pubmedbert_base_seed42_fold3
  pubmedbert_base_seed42_fold4
  pubmedbert_noweight_seed2024_fold0
  pubmedbert_noweight_seed2024_fold1
  pubmedbert_noweight_seed2024_fold2
  pubmedbert_noweight_seed2024_fold3
  pubmedbert_noweight_seed2024_fold4
  pubmedbert_noweight_seed42_fold0
  pubmedbert_noweight_seed42_fold1
  pubmedbert_noweight_seed42

/tmp/ipykernel_553/501030582.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/Data_Mining')


In [3]:
# === Cell 2: 工具函數 ===
import os, subprocess, tarfile, glob, json
import pandas as pd

def train_model(model, seed, epochs, bs, lr, tag_prefix, class_weight='none', script='train_bert.py'):
    for fold in range(5):
        cmd = (f'python src/{script} --model {model} '
               f'--fold {fold} --seed {seed} --epochs {epochs} --batch-size {bs} '
               f'--lr {lr} --max-length 512 '
               + (f'--class-weight {class_weight} ' if script == 'train_bert.py' else '')
               + f'--tag {tag_prefix}_fold{fold}')
        print('>>>', cmd)
        rc = os.system(cmd)
        assert rc == 0, f'{tag_prefix} fold {fold} failed'
    runs = sorted(glob.glob(f'outputs/bert_runs/{tag_prefix}_fold*/metrics.json'))
    df = pd.DataFrame([json.load(open(p)) for p in runs])
    print(f'\n{tag_prefix} per-fold val Macro F1:')
    print(df[['fold', 'val_macro_f1', 'train_secs']])
    mean = df.val_macro_f1.mean()
    print(f'{tag_prefix} mean OOF Macro F1: {mean:.4f}\n')
    return mean

def backup(label):
    src = '/content/Data_Mining/outputs/bert_runs'
    dst = f'/content/drive/MyDrive/Kaggle_backup/predictions_only_{label}.tar.gz'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    files_to_pack = []
    for run in sorted(os.listdir(src)):
        rd = os.path.join(src, run)
        if not os.path.isdir(rd): continue
        for pat in ['*.npy', '*.json', '*.csv']:
            files_to_pack += glob.glob(os.path.join(rd, pat))
    with tarfile.open(dst, 'w:gz') as tar:
        for f in files_to_pack:
            tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))
    size = subprocess.check_output(['du', '-h', dst]).decode().split()[0]
    print(f'Backup: {dst} ({size})')

def run_ensemble(patterns, tag):
    cmd = ['python', 'src/ensemble_predict.py'] + ['--bert-runs'] + patterns + ['--no-overlap-constraint', '--tag', tag]
    print('>>>', ' '.join(cmd))
    out = subprocess.run(cmd, cwd='/content/Data_Mining', capture_output=True, text=True)
    for line in out.stdout.split('\n'):
        if any(k in line for k in ['Found ', 'fold ', 'OOF Macro F1', 'macro avg', 'general path']):
            print(line)
    print()

print('工具函數已載入。')

工具函數已載入。


In [4]:
# === Cell 3: SciBERT noweight 5-fold ===
scibert_mean = train_model(
    model='allenai/scibert_scivocab_uncased',
    seed=42, epochs=4, bs=32, lr=2e-5,
    tag_prefix='scibert_noweight_seed42',
    class_weight='none',
)
backup('after_scibert')

if scibert_mean >= 0.640:
    print(f'✅ SciBERT OOF {scibert_mean:.4f} ≥ 0.640，繼續 Phase 2')
elif scibert_mean >= 0.620:
    print(f'⚠️ SciBERT OOF {scibert_mean:.4f} 中等，但加進 ensemble 試')
else:
    print(f'❌ SciBERT OOF {scibert_mean:.4f} 過低，考慮排除')

>>> python src/train_bert.py --model allenai/scibert_scivocab_uncased --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag scibert_noweight_seed42_fold0
>>> python src/train_bert.py --model allenai/scibert_scivocab_uncased --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag scibert_noweight_seed42_fold1
>>> python src/train_bert.py --model allenai/scibert_scivocab_uncased --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag scibert_noweight_seed42_fold2
>>> python src/train_bert.py --model allenai/scibert_scivocab_uncased --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag scibert_noweight_seed42_fold3
>>> python src/train_bert.py --model allenai/scibert_scivocab_uncased --fold 4 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag scibert_noweight_seed42_fold4


In [5]:
# === Cell 4a: DeBERTa fold 0 預跑（~14 分鐘）===
import os, json

print('=== DeBERTa fold 0 預跑（確認 lr 設定合適）===')
cmd = ('python src/train_bert.py '
       '--model microsoft/deberta-v3-base '
       '--fold 0 --seed 42 --epochs 4 --batch-size 24 '
       '--lr 1e-5 --max-length 512 --class-weight none '
       '--tag deberta_v3_noweight_seed42_fold0')
print('>>>', cmd)
rc = os.system(cmd)
assert rc == 0, 'DeBERTa fold 0 failed'

m = json.load(open('outputs/bert_runs/deberta_v3_noweight_seed42_fold0/metrics.json'))
print(f"\nDeBERTa fold 0 val Macro F1: {m['val_macro_f1']:.4f}")

if m['val_macro_f1'] >= 0.60:
    print('✅ DeBERTa 收斂良好，繼續 fold 1-4')
elif m['val_macro_f1'] >= 0.55:
    print('⚠️ DeBERTa 中等，繼續但結果可能不如預期')
else:
    print('❌ DeBERTa fold 0 < 0.55，**先停下來**！')
    print('   建議改用 lr=2e-5 重訓，刪掉 fold0 資料夾再跑')
    print('   (但通常 PubMed/醫學文本 DeBERTa lr=1e-5 應該 OK)')

=== DeBERTa fold 0 預跑（確認 lr 設定合適）===
>>> python src/train_bert.py --model microsoft/deberta-v3-base --fold 0 --seed 42 --epochs 4 --batch-size 24 --lr 1e-5 --max-length 512 --class-weight none --tag deberta_v3_noweight_seed42_fold0

DeBERTa fold 0 val Macro F1: 0.6527
✅ DeBERTa 收斂良好，繼續 fold 1-4


In [6]:
# === Cell 4b: DeBERTa fold 1-4 + 摘要 ===
import os, json, glob
import pandas as pd

# 跑剩餘 fold
for fold in range(1, 5):
    cmd = (f'python src/train_bert.py '
           f'--model microsoft/deberta-v3-base '
           f'--fold {fold} --seed 42 --epochs 4 --batch-size 24 '
           f'--lr 1e-5 --max-length 512 --class-weight none '
           f'--tag deberta_v3_noweight_seed42_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'DeBERTa fold {fold} failed'

# 摘要
runs = sorted(glob.glob('outputs/bert_runs/deberta_v3_noweight_seed42_fold*/metrics.json'))
df = pd.DataFrame([json.load(open(p)) for p in runs])
print('\nDeBERTa per-fold val Macro F1:')
print(df[['fold', 'val_macro_f1', 'train_secs']])
deberta_mean = df.val_macro_f1.mean()
print(f'\nDeBERTa mean OOF Macro F1: {deberta_mean:.4f}')

backup('after_deberta')

if deberta_mean >= 0.640:
    print(f'✅ DeBERTa OOF {deberta_mean:.4f} 健康，必加 ensemble')
elif deberta_mean >= 0.620:
    print(f'⚠️ DeBERTa OOF {deberta_mean:.4f} 中等')
else:
    print(f'❌ DeBERTa OOF {deberta_mean:.4f} 過低，考慮排除')

>>> python src/train_bert.py --model microsoft/deberta-v3-base --fold 1 --seed 42 --epochs 4 --batch-size 24 --lr 1e-5 --max-length 512 --class-weight none --tag deberta_v3_noweight_seed42_fold1
>>> python src/train_bert.py --model microsoft/deberta-v3-base --fold 2 --seed 42 --epochs 4 --batch-size 24 --lr 1e-5 --max-length 512 --class-weight none --tag deberta_v3_noweight_seed42_fold2
>>> python src/train_bert.py --model microsoft/deberta-v3-base --fold 3 --seed 42 --epochs 4 --batch-size 24 --lr 1e-5 --max-length 512 --class-weight none --tag deberta_v3_noweight_seed42_fold3
>>> python src/train_bert.py --model microsoft/deberta-v3-base --fold 4 --seed 42 --epochs 4 --batch-size 24 --lr 1e-5 --max-length 512 --class-weight none --tag deberta_v3_noweight_seed42_fold4

DeBERTa per-fold val Macro F1:
   fold  val_macro_f1  train_secs
0     0      0.652750  559.233673
1     1      0.641402  557.133682
2     2      0.637275  564.774422
3     3      0.657217  556.786028
4     4      0.636

In [7]:
# === Cell 5: Multi-label BCE PubMedBERT 5-fold ===
import os, json, glob
import pandas as pd

MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 42

for fold in range(5):
    # 注意：用的是 train_bert_multilabel.py（BCE 版本），沒有 --class-weight 參數
    cmd = (f'python src/train_bert_multilabel.py --model {MODEL} '
           f'--fold {fold} --seed {SEED} --epochs 4 --batch-size 32 '
           f'--lr 2e-5 --max-length 512 '
           f'--tag pubmedbert_bce_seed{SEED}_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'BCE fold {fold} failed'

# 摘要
runs = sorted(glob.glob('outputs/bert_runs/pubmedbert_bce_seed42_fold*/metrics.json'))
df = pd.DataFrame([json.load(open(p)) for p in runs])
print('\nBCE per-fold val Macro F1:')
print(df[['fold', 'val_macro_f1', 'train_secs']])
bce_mean = df.val_macro_f1.mean()
print(f'\nBCE mean OOF Macro F1: {bce_mean:.4f}')

backup('after_bce')

# 對比 noweight 看 BCE 有沒有突破
print(f'\n對比：')
print(f'  PubMedBERT noweight seed=42 (CE):  0.6524')
print(f'  PubMedBERT BCE seed=42:           {bce_mean:.4f}')
print(f'  差距: {bce_mean - 0.6524:+.4f}')

if bce_mean >= 0.660:
    print('\n✅ BCE 大成功！必加進 ensemble，v18 預期 LB +0.005~0.010')
elif bce_mean >= 0.640:
    print('\n⚠️ BCE 與 noweight 相當，加進來看 diversity 收益')
elif bce_mean >= 0.620:
    print('\n⚠️ BCE 略弱於 noweight，但 error pattern 可能不同，可加')
else:
    print('\n❌ BCE 明顯失敗，跳過')

>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_seed42_fold0
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_seed42_fold1
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_seed42_fold2
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_seed42_fold3
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 4 --seed 42 --e

In [8]:
# === Cell 6: 產生所有 ensemble 候選 ===
import subprocess

def run_ensemble(patterns, tag):
    cmd = ['python', 'src/ensemble_predict.py'] + ['--bert-runs'] + patterns + ['--no-overlap-constraint', '--tag', tag]
    print('>>>', ' '.join(cmd))
    out = subprocess.run(cmd, cwd='/content/Data_Mining', capture_output=True, text=True)
    if out.returncode != 0:
        print('STDERR:', out.stderr[:500])
        return
    for line in out.stdout.split('\n'):
        if any(k in line for k in ['Found ', 'fold ', 'OOF Macro F1', 'macro avg', 'general path', 'submission']):
            print(line)
    print()

# 候選 0: BCE only（單一模型最強 OOF）
run_ensemble([
    'outputs/bert_runs/pubmedbert_bce_seed*_fold*',
], 'final_bce_only')

# 候選 v15: 4 noweight (final_d 基礎) + SciBERT — 不含 seed=7（驗證為負）
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
    'outputs/bert_runs/scibert_noweight_seed*_fold*',
], 'v15_6models')

# 候選 v16: + DeBERTa
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
    'outputs/bert_runs/scibert_noweight_seed*_fold*',
    'outputs/bert_runs/deberta_v3_noweight_seed*_fold*',
], 'v16_7models')

# 候選 v17: v16 砍 large
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/scibert_noweight_seed*_fold*',
    'outputs/bert_runs/deberta_v3_noweight_seed*_fold*',
], 'v17_no_large')

# 候選 v18: v17 + BCE （★ 主打）
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/scibert_noweight_seed*_fold*',
    'outputs/bert_runs/deberta_v3_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbert_bce_seed*_fold*',
], 'v18_with_bce')

# 候選 v19: 全套（v18 + large）
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
    'outputs/bert_runs/scibert_noweight_seed*_fold*',
    'outputs/bert_runs/deberta_v3_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbert_bce_seed*_fold*',
], 'v19_all')

# 列出 md5 + 分布
import os, hashlib, pandas as pd
print('\n=== 所有候選 submission md5 + label dist ===')
for tag in ['final_bce_only', 'v15_6models', 'v16_7models', 'v17_no_large', 'v18_with_bce', 'v19_all']:
    p = f'outputs/submission_{tag}.csv'
    if os.path.exists(p):
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()[:10]
        df = pd.read_csv(p)
        dist = df['label'].value_counts(normalize=True).sort_index().round(3).to_dict()
        print(f'  {tag}: md5={h} dist={dist}')

# 全套備份
import tarfile, glob
src = '/content/Data_Mining/outputs/bert_runs'
dst = '/content/drive/MyDrive/Kaggle_backup/predictions_FINAL_0523.tar.gz'
files_to_pack = []
for run in sorted(os.listdir(src)):
    rd = os.path.join(src, run)
    if not os.path.isdir(rd): continue
    for pat in ['*.npy', '*.json', '*.csv']:
        files_to_pack += glob.glob(os.path.join(rd, pat))
with tarfile.open(dst, 'w:gz') as tar:
    for f in files_to_pack:
        tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))
print(f'\nFinal backup: {dst}')

# 下載 6 個候選 submission
from google.colab import files
for tag in ['final_bce_only', 'v15_6models', 'v16_7models', 'v17_no_large', 'v18_with_bce', 'v19_all']:
    p = f'/content/Data_Mining/outputs/submission_{tag}.csv'
    if os.path.exists(p):
        files.download(p)

>>> python src/ensemble_predict.py --bert-runs outputs/bert_runs/pubmedbert_bce_seed*_fold* --no-overlap-constraint --tag final_bce_only
Found 5 BERT run dirs:
  fold 0: 1 run(s), val macro F1 = 0.7057
  fold 1: 1 run(s), val macro F1 = 0.6970
  fold 2: 1 run(s), val macro F1 = 0.6947
  fold 3: 1 run(s), val macro F1 = 0.7029
  fold 4: 1 run(s), val macro F1 = 0.6784
BERT OOF Macro F1: 0.6960
general pathological conditions     0.7065    0.4682    0.5631      4334
                      macro avg     0.6828    0.7259    0.6960     12994
Ensemble OOF Macro F1: 0.6960
general pathological conditions     0.7065    0.4682    0.5631      4334
                      macro avg     0.6828    0.7259    0.6960     12994
Wrote final submission: /content/Data_Mining/outputs/submission_final_bce_only.csv
   submission  train

>>> python src/ensemble_predict.py --bert-runs outputs/bert_runs/pubmedbert_noweight_seed42_fold* outputs/bert_runs/pubmedbert_noweight_seed2024_fold* outputs/bert_runs/biobert_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
%cd /content/Data_Mining
!git pull

/content
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 13 (delta 10), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 7.22 KiB | 1.20 MiB/s, done.
From https://github.com/eric20041027/Data_Mining
   e76303d..521dc60  main       -> origin/main
Updating e76303d..521dc60
Fast-forward
 CHANGELOG.md                 | 100 ++++++++++++++++++++++++++++++++++---------
 README.md                    |  21 +++++++--
 src/train_bert_multilabel.py |  12 +++++-
 src/utils.py                 |  25 +++++++++++
 4 files changed, 133 insertions(+), 25 deletions(-)


In [10]:
import os
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 42
for fold in range(5):
    cmd = (f'python src/train_bert_multilabel.py --model {MODEL} '
           f'--fold {fold} --seed {SEED} --epochs 4 --batch-size 32 '
           f'--lr 2e-5 --max-length 512 '
           f'--tag pubmedbert_bce_grouped_seed{SEED}_fold{fold}')
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'BCE grouped fold {fold} failed'

# OOF 摘要
import json, glob
import pandas as pd
runs = sorted(glob.glob('outputs/bert_runs/pubmedbert_bce_grouped_seed42_fold*/metrics.json'))
df = pd.DataFrame([json.load(open(p)) for p in runs])
print('\n=== BCE-Grouped per-fold val Macro F1 ===')
print(df[['fold', 'val_macro_f1', 'train_secs']])
bce_grouped_mean = df.val_macro_f1.mean()
print(f'\nBCE-Grouped mean OOF Macro F1: {bce_grouped_mean:.4f}')

print(f'\n對比：')
print(f'  BCE (StratifiedKFold leaky):     OOF 0.6957  LB 0.596 ❌')
print(f'  BCE-Grouped (no leak):           OOF {bce_grouped_mean:.4f}')
print(f'  PubMedBERT noweight (CE):        OOF 0.6524  LB 0.641')
print(f'  final_d (4-model ensemble):      OOF 0.6592  LB 0.646')

# 備份
import tarfile, glob
src = '/content/Data_Mining/outputs/bert_runs'
dst = '/content/drive/MyDrive/Kaggle_backup/predictions_bce_grouped.tar.gz'
files_to_pack = []
for run in sorted(os.listdir(src)):
    rd = os.path.join(src, run)
    if not os.path.isdir(rd): continue
    for pat in ['*.npy', '*.json', '*.csv']:
        files_to_pack += glob.glob(os.path.join(rd, pat))
with tarfile.open(dst, 'w:gz') as tar:
    for f in files_to_pack:
        tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))
print(f'\nBackup: {dst}')

>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_grouped_seed42_fold0
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_grouped_seed42_fold1
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_grouped_seed42_fold2
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 --tag pubmedbert_bce_grouped_seed42_fold3
>>> python src/train_bert_multilabel.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract

In [13]:
from google.colab import files
files.download('/content/Data_Mining/outputs/submission_final_bce_grouped_only.csv')

FileNotFoundError: Cannot find file: /content/Data_Mining/outputs/submission_final_bce_grouped_only.csv

In [12]:
import subprocess

# 純 BCE-Grouped
cmd = ['python', 'src/ensemble_predict.py',
       '--bert-runs', 'outputs/bert_runs/pubmedbert_bce_grouped_seed*_fold*',
       '--no-overlap-constraint',
       '--tag', 'final_bce_grouped_only']
out = subprocess.run(cmd, cwd='/content/Data_Mining', capture_output=True, text=True)
for line in out.stdout.split('\n'):
    if any(k in line for k in ['Found ', 'fold ', 'OOF Macro F1', 'macro avg', 'general path', 'submission']):
        print(line)

# 下載
from google.colab import files
files.download('/content/Data_Mining/outputs/submission_final_bce_grouped_only.csv')

Found 5 BERT run dirs:


FileNotFoundError: Cannot find file: /content/Data_Mining/outputs/submission_final_bce_grouped_only.csv

In [14]:
!ls /content/Data_Mining/outputs/

baseline_metrics.json	       submission_v15_6models.csv
bert_runs		       submission_v16_7models.csv
fold_assignment.csv	       submission_v17_no_large.csv
oof_tfidf_logreg.npy	       submission_v18_with_bce.csv
submission_final_bce_only.csv  submission_v19_all.csv
submissions		       test_tfidf_logreg.npy
